# Q-learning 完整教学演示：纯 PyTorch 张量版

这一份 notebook 可以作为**唯一主讲版本**。它从零定义环境，沿着“环境 → 状态 → 动作 → 奖励 → 下一状态 → Q 更新”完整展开，最后训练、评估、保存并重新加载模型。

**本例只使用**：Python 标准库、PyTorch、Matplotlib。

**不使用**：Gym、NumPy、神经网络、反向传播、优化器。

**适合读者**：第一次系统学习 Q-learning，或准备从表格型 Q-learning 过渡到 DQN 的学习者。

**学完能够**：

- 逐项解释环境、状态、动作、奖励、策略、回合和 Q 值；
- 对照代码讲清一次 `env.step()` 和一次 Q 更新；
- 看懂探索率、TD 误差、成功率、Q 热力图和策略箭头；
- 理解表格型 Q-learning 的“训练模型”就是学到的 Q 表；
- 保存、加载并验证一个 PyTorch Q-learning 检查点。


## 教学路线

1. 总览完整概念链
2. 定义环境与奖励规则
3. 将位置编码成状态编号
4. 定义四个离散动作
5. 用三张张量表表示环境转移
6. 拆解一次真实交互
7. 用 ε-greedy 选择动作
8. 手算并执行一次 Q 更新
9. 运行带详细注释的完整训练循环
10. 可视化训练指标、结果、Q 表和策略演化
11. 关闭探索并逐步验收路线
12. 保存、加载并检查训练模型

| 概念 | 本例中的定义 | PyTorch 表示 |
|---|---|---|
| 环境 | 5×5 网格、墙、陷阱、终点、转移规则 | `TorchGridWorld` |
| 状态 $s$ | 智能体当前所在格子的编号 | `torch.long` 标量 |
| 动作 $a$ | 上、右、下、左 | `torch.long` 标量，0~3 |
| 奖励 $r$ | 环境对刚发生转移的即时反馈 | `torch.float32` 标量 |
| 下一状态 $s'$ | 执行动作后的新位置编号 | `torch.long` 标量 |
| 终止 `done` | 是否到达终点或掉入陷阱 | `torch.bool` 标量 |
| Q 表 | 每个状态—动作组合的长期价值估计 | `[24, 4]` 张量 |
| 策略 | 从当前 Q 表选择动作的方法 | ε-greedy / `argmax` |


In [ ]:
# ===== 0. 全局设置：依赖、随机种子、字体、颜色和图片目录 =====
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import html
import os

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib import colors, font_manager, patches
from IPython.display import HTML, display

SEED = 7
torch.manual_seed(SEED)

# 这个 Q 表只有 24×4=96 个数，CPU 比 GPU 更适合逐步教学。
DEVICE = torch.device("cpu")

# 自动选择可用中文字体。
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
font_candidates = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Arial Unicode MS", "DejaVu Sans"]
CJK_FONT = next((name for name in font_candidates if name in available_fonts), "DejaVu Sans")

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [CJK_FONT, "DejaVu Sans"],
    "axes.unicode_minus": False,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
})

# 色盲友好、适合投影的统一配色。
C = {
    "ink": "#264653", "teal": "#2A9D8F", "gold": "#E9C46A",
    "orange": "#F4A261", "coral": "#E76F51", "blue": "#0072B2",
    "sky": "#56B4E9", "green": "#009E73", "gray": "#9AA0A6",
    "light": "#F5F7F9", "wall": "#59636E", "trap": "#D55E00",
}

FIG_DIR = Path("figures_pytorch_full")
FIG_DIR.mkdir(exist_ok=True)

def save_figure(fig, name):
    # 每张图同时导出 300 DPI PNG 和矢量 PDF，方便 notebook、PPT 和打印使用。
    fig.savefig(FIG_DIR / f"{name}.png", dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight", facecolor="white")

print(f"PyTorch {torch.__version__} | CUDA 可用: {torch.cuda.is_available()} | 本演示设备: {DEVICE}")
print(f"随机种子: {SEED} | 中文字体: {CJK_FONT} | 图片目录: {FIG_DIR.resolve()}")


## 1. 总览：环境—状态—动作—奖励完整闭环

一次强化学习交互包含两个角色：

- **智能体**读取状态、选择动作、更新 Q 表；
- **环境**接收动作、执行世界规则、返回奖励和下一状态。

每次训练都反复执行同一条链。图中的代码片段与后续单元格一一对应。


In [ ]:
def draw_full_chain():
    fig, ax = plt.subplots(figsize=(15.5, 4.4))
    ax.set_xlim(0, 15.5); ax.set_ylim(0, 4.4); ax.axis("off")

    boxes = [
        (0.2, "① 环境", "网格与转移规则\nTorchGridWorld", C["sky"]),
        (2.35, "② 状态 s_t", "当前位置编号\ntorch.long", "#B9DCEB"),
        (4.5, "③ 策略 π", "ε-greedy\n探索 / 利用", C["gold"]),
        (6.65, "④ 动作 a_t", "上 / 右 / 下 / 左\n0 / 1 / 2 / 3", C["orange"]),
        (8.8, "⑤ env.step", "查转移张量\n执行世界规则", C["sky"]),
        (10.95, "⑥ 环境反馈", "r_(t+1), s_(t+1), done\n三个张量", C["coral"]),
        (13.1, "⑦ Q 更新", "TD 目标与 TD 误差\n更新 Q[s_t,a_t]", C["teal"]),
    ]

    for x, title, subtitle, color in boxes:
        rect = patches.FancyBboxPatch(
            (x, 1.72), 1.85, 1.35,
            boxstyle="round,pad=0.08,rounding_size=0.12",
            facecolor=color, edgecolor="white", linewidth=1.6
        )
        ax.add_patch(rect)
        ax.text(x+0.925, 2.63, title, ha="center", va="center", weight="bold", color=C["ink"])
        ax.text(x+0.925, 2.12, subtitle, ha="center", va="center", fontsize=8.7, color=C["ink"])

    for x in [2.05, 4.2, 6.35, 8.5, 10.65, 12.8]:
        ax.annotate("", xy=(x+0.25, 2.40), xytext=(x, 2.40),
                    arrowprops=dict(arrowstyle="->", lw=2, color=C["ink"]))

    ax.annotate(
        "done=False：把 s_(t+1) 作为下一步的 s_t，继续循环",
        xy=(1.1, 1.64), xytext=(14.0, 0.52), ha="center", va="center", color=C["ink"],
        arrowprops=dict(arrowstyle="->", lw=2, color=C["ink"], connectionstyle="arc3,rad=-0.18")
    )
    ax.text(7.75, 3.75, "智能体负责选择和学习；环境负责响应和反馈", ha="center", fontsize=12, weight="bold", color=C["ink"])
    ax.set_title("图 1｜Q-learning 的完整单步闭环", fontsize=15, pad=8)
    fig.tight_layout()
    save_figure(fig, "01_full_concept_chain")
    return fig

draw_full_chain();


## 2. 环境：先定义“世界如何运转”

环境包含两类信息：

1. **空间结构**：网格、边界、墙、起点、陷阱、终点；
2. **转移与奖励规则**：执行某个动作后去哪里、得到多少奖励、回合是否结束。

环境并不决定智能体应该选什么动作；它只诚实回答：“你这样做以后，发生了什么？”


In [ ]:
# ===== 1. 环境定义：所有关键步骤都带注释 =====
@dataclass
class StepResult:
    next_state: torch.Tensor  # 下一状态 s'
    reward: torch.Tensor      # 即时奖励 r
    done: torch.Tensor        # 是否终止
    info: dict                # 只用于解释和可视化，不参与学习


class TorchGridWorld:
    # 动作编号、中文名、箭头和位移必须保持同一顺序。
    ACTIONS = ("上", "右", "下", "左")
    ARROWS = ("↑", "→", "↓", "←")
    DELTAS = ((-1, 0), (0, 1), (1, 0), (0, -1))

    def __init__(self, device="cpu"):
        self.device = torch.device(device)
        self.rows, self.cols = 5, 5
        self.start = (4, 0)
        self.goal = (0, 4)
        self.traps = {(1, 3), (3, 2)}
        self.walls = {(2, 2)}

        # 墙不是可进入状态，因此 25 个格子中只有 24 个有效状态。
        self.positions = [
            (r, c) for r in range(self.rows) for c in range(self.cols)
            if (r, c) not in self.walls
        ]
        self.pos_to_state = {pos: i for i, pos in enumerate(self.positions)}
        self.state_to_pos = {i: pos for pos, i in self.pos_to_state.items()}
        self.n_states = len(self.positions)
        self.n_actions = len(self.ACTIONS)

        # 三张环境转移表：输入 (state, action)，得到 next_state、reward、done。
        self.next_states = torch.empty((self.n_states, self.n_actions), dtype=torch.long, device=self.device)
        self.rewards = torch.empty((self.n_states, self.n_actions), dtype=torch.float32, device=self.device)
        self.dones = torch.empty((self.n_states, self.n_actions), dtype=torch.bool, device=self.device)
        self._build_transition_tensors()

        # 环境内部保存智能体当前状态；reset() 会把它放回起点。
        self.state = torch.tensor(self.pos_to_state[self.start], dtype=torch.long, device=self.device)

    def _build_transition_tensors(self):
        # 遍历每个有效状态和四个动作，一次性编码全部环境规则。
        for state, pos in self.state_to_pos.items():
            for action, (dr, dc) in enumerate(self.DELTAS):
                candidate = (pos[0] + dr, pos[1] + dc)

                # 越界或撞墙：位置不变，并得到 -1 奖励。
                collision = (
                    candidate[0] < 0 or candidate[0] >= self.rows
                    or candidate[1] < 0 or candidate[1] >= self.cols
                    or candidate in self.walls
                )
                new_pos = pos if collision else candidate

                # 奖励评价“刚刚发生的转移”，不是状态本身的永久属性。
                if collision:
                    reward = -1.0
                elif new_pos == self.goal:
                    reward = 10.0
                elif new_pos in self.traps:
                    reward = -10.0
                else:
                    reward = -0.1

                done = new_pos == self.goal or new_pos in self.traps
                self.next_states[state, action] = self.pos_to_state[new_pos]
                self.rewards[state, action] = reward
                self.dones[state, action] = done

    def reset(self):
        # 每个回合从同一个起点开始。
        self.state = torch.tensor(self.pos_to_state[self.start], dtype=torch.long, device=self.device)
        return self.state.clone()

    def step(self, action):
        # action 可以是 Python int，也可以是 torch.long 标量。
        action = torch.as_tensor(action, dtype=torch.long, device=self.device)
        old_state = self.state.clone()

        # 环境执行一步就是查三张转移张量表。
        next_state = self.next_states[old_state, action]
        reward = self.rewards[old_state, action]
        done = self.dones[old_state, action]

        # 更新环境内部位置，让下一次 step 从新状态出发。
        self.state = next_state.clone()
        return StepResult(
            next_state=next_state.clone(), reward=reward.clone(), done=done.clone(),
            info={
                "old_state": old_state.item(),
                "old_pos": self.state_to_pos[old_state.item()],
                "new_pos": self.state_to_pos[next_state.item()],
                "action_name": self.ACTIONS[action.item()],
            },
        )


env = TorchGridWorld(DEVICE)
print(f"有效状态数: {env.n_states} | 动作数: {env.n_actions} | Q 表形状: ({env.n_states}, {env.n_actions})")


In [ ]:
# ===== 2. 环境与奖励规则可视化 =====
GRID_CMAP = colors.ListedColormap(["#F7F8FA", "#A7D8C9", "#F3D37A", "#E98770", "#59636E"])
GRID_NORM = colors.BoundaryNorm(torch.arange(-0.5, 5.5, 1).tolist(), GRID_CMAP.N)

def base_grid(env):
    grid = torch.zeros((env.rows, env.cols), dtype=torch.long)
    grid[env.start] = 1
    grid[env.goal] = 2
    for pos in env.traps: grid[pos] = 3
    for pos in env.walls: grid[pos] = 4
    return grid

def format_grid_axes(ax, env, title=""):
    ax.set_xticks(range(env.cols), [f"列 {i}" for i in range(env.cols)])
    ax.set_yticks(range(env.rows), [f"行 {i}" for i in range(env.rows)])
    ax.set_xticks(torch.arange(-0.5, env.cols, 1).tolist(), minor=True)
    ax.set_yticks(torch.arange(-0.5, env.rows, 1).tolist(), minor=True)
    ax.grid(which="minor", color="white", linewidth=2)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 5.2), gridspec_kw={"width_ratios": [1.25, 1]})

# 左：空间结构
axes[0].imshow(base_grid(env), cmap=GRID_CMAP, norm=GRID_NORM)
format_grid_axes(axes[0], env, "环境空间结构")
labels = {env.start: "起点\nS", env.goal: "终点\nG", **{p: "陷阱\nX" for p in env.traps}, **{p: "墙" for p in env.walls}}
for pos, label in labels.items():
    axes[0].text(pos[1], pos[0], label, ha="center", va="center", weight="bold",
                 color="white" if pos in env.walls else C["ink"])

# 右：奖励设计
events = ["到达终点", "普通移动", "撞墙/越界", "进入陷阱"]
rewards = torch.tensor([10.0, -0.1, -1.0, -10.0])
bar_colors = [C["green"], C["sky"], C["orange"], C["trap"]]
bars = axes[1].barh(events, rewards, color=bar_colors, height=0.58)
axes[1].axvline(0, color=C["ink"], lw=1)
for bar, value in zip(bars, rewards):
    axes[1].text(value.item() + (0.35 if value >= 0 else -0.35), bar.get_y()+bar.get_height()/2,
                 f"{value.item():g}", ha="left" if value >= 0 else "right", va="center", weight="bold")
axes[1].set_xlim(-12, 12); axes[1].set_xlabel("即时奖励 r")
axes[1].set_title("奖励函数：告诉智能体什么值得做")
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].grid(axis="x", alpha=0.16)

fig.suptitle("图 2｜环境 = 空间结构 + 转移规则 + 奖励规则", fontsize=14, weight="bold")
fig.tight_layout()
save_figure(fig, "02_environment_and_reward")
plt.show()


## 3. 状态：智能体如何描述“我在哪里”

状态必须包含做决策所需的信息。本例环境是静态网格，因此当前位置 `(行, 列)` 足够描述状态。

Q 表需要整数索引，所以把每个有效位置映射到 `0~23`。墙不是可进入位置，因此没有状态编号。

动作则是智能体可以发给环境的四条指令。动作编号本身没有“好坏”，好坏取决于当前状态和后续奖励。


In [ ]:
# ===== 3. 状态编码图 + 动作空间图 =====
state_grid = torch.full((env.rows, env.cols), -1, dtype=torch.long)
for state, pos in env.state_to_pos.items():
    state_grid[pos] = state

fig, axes = plt.subplots(1, 2, figsize=(11.8, 5.2))

# 左：位置到状态编号
cmap_states = plt.cm.Blues.copy()
cmap_states.set_under(C["wall"])
axes[0].imshow(state_grid, cmap=cmap_states, vmin=0, vmax=env.n_states-1)
format_grid_axes(axes[0], env, "位置 (row, col) → 状态编号 state")
for r in range(env.rows):
    for c in range(env.cols):
        value = state_grid[r, c].item()
        axes[0].text(c, r, "墙" if value < 0 else f"s={value}", ha="center", va="center",
                     color="white" if value < 0 or value > 15 else C["ink"], weight="bold", fontsize=9)

# 右：动作空间
axes[1].set_xlim(-1.7, 1.7); axes[1].set_ylim(-1.7, 1.7); axes[1].axis("off")
axes[1].add_patch(patches.Circle((0, 0), 0.34, facecolor=C["teal"], edgecolor="white", lw=2))
axes[1].text(0, 0, "智能体", ha="center", va="center", color="white", weight="bold")
for action, ((dr, dc), name, arrow) in enumerate(zip(env.DELTAS, env.ACTIONS, env.ARROWS)):
    x, y = dc*1.15, -dr*1.15
    axes[1].annotate("", xy=(x, y), xytext=(dc*0.4, -dr*0.4),
                     arrowprops=dict(arrowstyle="-|>", lw=4, color=[C["blue"], C["orange"], C["green"], C["coral"]][action]))
    axes[1].text(x*1.15, y*1.15, f"动作 {action}\n{name} {arrow}", ha="center", va="center", weight="bold")
axes[1].set_title("动作空间 A = {0,1,2,3}")

fig.suptitle("图 3｜状态回答“在哪里”，动作回答“能做什么”", fontsize=14, weight="bold")
fig.tight_layout()
save_figure(fig, "03_state_and_action")
plt.show()


## 4. 环境转移张量：`env.step()` 内部究竟查了什么

三张大小均为 `[24, 4]` 的张量完整描述了这个确定性环境：

```text
(state, action)
      ├── next_states[state, action] → 下一状态
      ├── rewards[state, action]     → 即时奖励
      └── dones[state, action]       → 是否终止
```

这三张表是**环境模型**；Q 表是智能体通过经验学到的**价值模型**。不要把二者混淆。


In [ ]:
# ===== 4. 三张环境转移张量的全局可视化 =====
fig, axes = plt.subplots(3, 1, figsize=(13.5, 8.6), sharex=True, layout="constrained")

panels = [
    (env.next_states.T.float(), "下一状态 next_states[a, s]", "viridis"),
    (env.rewards.T, "即时奖励 rewards[a, s]", "RdYlGn"),
    (env.dones.T.float(), "终止标记 dones[a, s]", colors.ListedColormap(["#E8EDF2", C["coral"]])),
]
for ax, (matrix, title, cmap) in zip(axes, panels):
    im = ax.imshow(matrix, aspect="auto", cmap=cmap)
    ax.set_yticks(range(env.n_actions), [f"{i}/{name}" for i, name in enumerate(env.ACTIONS)])
    ax.set_ylabel("动作 a")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.78, pad=0.015)
axes[-1].set_xticks(range(env.n_states), [str(i) for i in range(env.n_states)], fontsize=8)
axes[-1].set_xlabel("状态 s")
fig.suptitle("图 4｜环境的三张查找表：每一列是状态，每一行是动作", fontsize=14, weight="bold")
save_figure(fig, "04_transition_tensors")
plt.show()

# 单独打印起点的四种动作，方便对照查表过程。
start_state = env.pos_to_state[env.start]
rows = []
for action in range(env.n_actions):
    ns = env.next_states[start_state, action].item()
    rows.append([
        action, env.ACTIONS[action], env.state_to_pos[start_state],
        env.state_to_pos[ns], env.rewards[start_state, action].item(), env.dones[start_state, action].item()
    ])
headers = ["动作编号", "动作", "当前位置", "下一位置", "奖励", "done"]
table = "<table style='border-collapse:collapse'><tr>" + "".join(
    f"<th style='padding:7px 10px;border:1px solid #ddd;background:#264653;color:white'>{h}</th>" for h in headers
) + "</tr>" + "".join(
    "<tr>" + "".join(f"<td style='padding:7px 10px;border:1px solid #ddd;text-align:center'>{html.escape(str(v))}</td>" for v in row) + "</tr>"
    for row in rows
) + "</table>"
display(HTML("<b>起点 state=19 的四种环境响应：</b><br><br>" + table))


## 5. 拆解一次真实交互

让智能体从起点执行动作 `0/上`。这一步会生成一条经验：

$$
(s_t, a_t, r_{t+1}, s_{t+1}, done)
$$

Q-learning 不需要看到整张地图才能更新；每次只需要这五项信息。


In [ ]:
# ===== 5. 一次真实交互及其可视化 =====
state = env.reset()                       # ① 环境给出当前状态
action = torch.tensor(0, dtype=torch.long)  # ② 智能体选择“上”
result = env.step(action)                 # ③ 环境执行动作并返回反馈

experience = {
    "s_t 当前状态": f"{state.item()} / {result.info['old_pos']}",
    "a_t 动作": f"{action.item()} / {result.info['action_name']}",
    "r_(t+1) 奖励": f"{result.reward.item():.1f}",
    "s_(t+1) 下一状态": f"{result.next_state.item()} / {result.info['new_pos']}",
    "done": result.done.item(),
}

fig, axes = plt.subplots(1, 3, figsize=(14.3, 4.8), gridspec_kw={"width_ratios": [1, 1, 1.3]})
for ax, pos, title in zip(
    axes[:2], [result.info["old_pos"], result.info["new_pos"]],
    [f"执行前：s_t={state.item()}", f"执行后：s_(t+1)={result.next_state.item()}"]
):
    ax.imshow(base_grid(env), cmap=GRID_CMAP, norm=GRID_NORM)
    format_grid_axes(ax, env, title)
    ax.scatter(pos[1], pos[0], s=520, color=C["blue"], edgecolor="white", lw=2, zorder=3)
    ax.text(pos[1], pos[0], "智能体", ha="center", va="center", color="white", weight="bold", fontsize=9, zorder=4)

# 第三个面板直接画出经验元组。
axes[2].axis("off")
axes[2].add_patch(patches.FancyBboxPatch((0.05, 0.12), 0.9, 0.76, transform=axes[2].transAxes,
                    boxstyle="round,pad=0.03", facecolor=C["light"], edgecolor=C["teal"], lw=2))
for i, (key, value) in enumerate(experience.items()):
    axes[2].text(0.12, 0.78-i*0.14, key, transform=axes[2].transAxes, weight="bold", color=C["ink"])
    axes[2].text(0.62, 0.78-i*0.14, str(value), transform=axes[2].transAxes, color=C["coral"] if "奖励" in key else C["ink"])
axes[2].set_title("一条经验 transition")

fig.suptitle("图 5｜状态 → 动作 → 环境响应 → 奖励与下一状态", fontsize=14, weight="bold")
fig.tight_layout()
save_figure(fig, "05_single_transition")
plt.show()


## 6. 策略：如何在探索与利用之间选择动作

训练初期 Q 表全为零，智能体并不知道哪条路好，因此必须探索。

**ε-greedy 策略**：

- 以概率 $\varepsilon$ 随机选择动作——探索未知选择；
- 以概率 $1-\varepsilon$ 选择 Q 值最大的动作——利用已有知识；
- 如果多个动作并列最大，随机打破并列，避免固定偏向动作 0。


In [ ]:
# ===== 6. ε-greedy 动作选择：同时返回是否发生探索 =====
def select_action(q_row, epsilon, generator):
    # 先用一个 [0,1) 随机数决定探索还是利用。
    explored = torch.rand((), generator=generator).item() < epsilon

    if explored:
        # 探索：四个动作等概率随机选择。
        action = torch.randint(len(q_row), (), generator=generator)
    else:
        # 利用：选择当前 Q 值最大的动作。
        best_actions = torch.where(q_row == q_row.max())[0]
        # 随机打破并列最大值，避免初始全零 Q 表总选动作 0。
        pick = torch.randint(len(best_actions), (), generator=generator)
        action = best_actions[pick]
    return action, explored


# 用一行示例 Q 值解释概率分配。
sample_q = torch.tensor([1.0, 2.5, 0.5, -1.0])
epsilon_demo = 0.40
best_action = sample_q.argmax().item()
theory_prob = torch.full((4,), epsilon_demo / 4)
theory_prob[best_action] += 1 - epsilon_demo

# 用 PyTorch 随机采样验证理论概率。
generator = torch.Generator().manual_seed(SEED)
counts = torch.zeros(4)
for _ in range(5000):
    a, _ = select_action(sample_q, epsilon_demo, generator)
    counts[a] += 1
empirical_prob = counts / counts.sum()

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.2))
actions_text = [f"{i}/{name}" for i, name in enumerate(env.ACTIONS)]
bars = axes[0].bar(actions_text, sample_q, color=[C["gray"], C["green"], C["gray"], C["gray"]])
for bar, value in zip(bars, sample_q):
    axes[0].text(bar.get_x()+bar.get_width()/2, value.item()+0.08, f"{value.item():.1f}", ha="center")
axes[0].set_title("当前状态的四个 Q 值")
axes[0].set_ylabel("Q(s,a)"); axes[0].axhline(0, color=C["ink"], lw=0.8)

x = torch.arange(4)
axes[1].bar(x-0.18, theory_prob*100, width=0.36, color=C["blue"], label="理论概率")
axes[1].bar(x+0.18, empirical_prob*100, width=0.36, color=C["gold"], label="5000 次采样")
axes[1].set_xticks(x, actions_text); axes[1].set_ylabel("选择概率 (%)")
axes[1].set_title(f"ε={epsilon_demo:.1f}：探索 40%，利用 60%")
axes[1].legend(frameon=False)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.15)
fig.suptitle("图 6｜ε-greedy 如何把 Q 值变成动作选择", fontsize=14, weight="bold")
fig.tight_layout()
save_figure(fig, "06_epsilon_greedy")
plt.show()


## 7. 学习：一条经验如何改变一个 Q 值

Q 值的含义：

> $Q(s,a)$ 是在状态 $s$ 执行动作 $a$ 后，未来能够获得的折扣累计奖励估计。

Q-learning 更新公式：

$$
\underbrace{Q(s,a)}_{新值}
\leftarrow
\underbrace{Q(s,a)}_{旧估计}
+\alpha\left[
\underbrace{r+\gamma(1-done)\max_{a'}Q(s',a')}_{TD\ 目标}
-\underbrace{Q(s,a)}_{旧估计}
\right]
$$

- $\alpha$：学习率，控制一次经验的改写幅度；
- $\gamma$：折扣因子，控制对未来奖励的重视程度；
- `done=True` 时未来价值必须为零。


In [ ]:
# ===== 7. 用具体数字执行一次 TD 更新 =====
old_q = torch.tensor(1.20)
reward = torch.tensor(-0.10)
next_best_q = torch.tensor(2.00)
alpha = 0.30
gamma = 0.95
done = torch.tensor(False)

# 终止时不能继续估计未来；这里 done=False，所以保留下一状态最大 Q 值。
future = torch.tensor(0.0) if done.item() else next_best_q
td_target = reward + gamma * future
td_error = td_target - old_q
new_q = old_q + alpha * td_error

print(f"TD 目标 = {reward.item():.2f} + {gamma:.2f} × {future.item():.2f} = {td_target.item():.2f}")
print(f"TD 误差 = {td_target.item():.2f} - {old_q.item():.2f} = {td_error.item():.2f}")
print(f"新 Q 值 = {old_q.item():.2f} + {alpha:.2f} × {td_error.item():.2f} = {new_q.item():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.1), gridspec_kw={"width_ratios": [1.45, 1]})

# 左：计算流程
axes[0].axis("off")
flow_boxes = [
    (0.02, "即时奖励", f"r = {reward.item():.2f}", C["coral"]),
    (0.27, "未来价值", f"γ max Q = {(gamma*future).item():.2f}", C["gold"]),
    (0.52, "TD 目标", f"target = {td_target.item():.2f}", C["sky"]),
    (0.77, "更新结果", f"new Q = {new_q.item():.2f}", C["teal"]),
]
for x0, title, value, color in flow_boxes:
    axes[0].add_patch(patches.FancyBboxPatch((x0, 0.34), 0.19, 0.34, transform=axes[0].transAxes,
                      boxstyle="round,pad=0.02", facecolor=color, edgecolor="white"))
    axes[0].text(x0+0.095, 0.57, title, transform=axes[0].transAxes, ha="center", weight="bold", color=C["ink"])
    axes[0].text(x0+0.095, 0.43, value, transform=axes[0].transAxes, ha="center", color=C["ink"])
for x0 in [0.22, 0.47, 0.72]:
    axes[0].annotate("", xy=(x0+0.04, 0.51), xytext=(x0, 0.51), xycoords=axes[0].transAxes,
                     arrowprops=dict(arrowstyle="->", lw=2, color=C["ink"]))
axes[0].set_title("从环境反馈到 Q 更新")

# 右：旧值、目标、新值的位置关系
values = torch.stack([old_q, td_target, new_q])
bars = axes[1].bar(["旧 Q", "TD 目标", "新 Q"], values, color=[C["gray"], C["coral"], C["teal"]])
for bar, value in zip(bars, values):
    axes[1].text(bar.get_x()+bar.get_width()/2, value.item()+0.04, f"{value.item():.2f}", ha="center", weight="bold")
axes[1].set_ylim(0, 2.15); axes[1].set_title("新 Q 只向目标移动 α=30%")
axes[1].spines[["top", "right"]].set_visible(False); axes[1].grid(axis="y", alpha=0.15)

fig.suptitle("图 7｜一条经验只更新 Q 表中的一个单元格", fontsize=14, weight="bold")
fig.tight_layout()
save_figure(fig, "07_td_update")
plt.show()


## 8. 完整训练循环：把概念链翻译成代码

下面的函数是本 notebook 的核心。注释中的 ①~⑧ 与图 1 的概念链对应。

额外记录以下信息用于教学可视化：

- 每回合累计奖励、步数、成功与否、ε；
- 每回合平均绝对 TD 误差；
- 每回合探索动作比例；
- 访问每个状态的次数；
- 0、10、50、200、800 回合时的 Q 表快照；
- 回合结局：终点、陷阱或超时。


In [ ]:
# ===== 8. 带逐步注释和完整日志的 PyTorch Q-learning =====
def train_q_learning(
    env,
    episodes=800,
    alpha=0.15,
    gamma=0.95,
    epsilon_start=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.992,
    max_steps=100,
    seed=SEED,
):
    generator = torch.Generator(device="cpu").manual_seed(seed)

    # Q[state, action]：24 个状态 × 4 个动作，初始时完全未知，所以全为 0。
    Q = torch.zeros((env.n_states, env.n_actions), dtype=torch.float32, device=DEVICE)

    # 训练日志：预先分配张量，避免在循环中不断扩展 Python 列表。
    returns = torch.empty(episodes)
    lengths = torch.empty(episodes, dtype=torch.long)
    successes = torch.empty(episodes, dtype=torch.bool)
    epsilons = torch.empty(episodes)
    mean_abs_td = torch.empty(episodes)
    explore_ratios = torch.empty(episodes)
    outcomes = torch.empty(episodes, dtype=torch.long)  # 1=终点，-1=陷阱，0=超时
    visits = torch.zeros(env.n_states, dtype=torch.long)

    checkpoint_episodes = {0, 10, 50, 200, episodes}
    snapshots = {0: Q.clone()}
    epsilon = epsilon_start

    # 表格型更新直接修改 Q 张量，不需要构建自动微分计算图。
    with torch.no_grad():
        for episode in range(1, episodes + 1):
            # ① 环境 reset：给出起始状态 s₀。
            state = env.reset()
            total_reward = torch.tensor(0.0)
            td_error_sum = torch.tensor(0.0)
            explore_count = 0
            outcome = 0

            for step in range(1, max_steps + 1):
                visits[state] += 1

                # ② 读取 Q[state]；③ 按 ε-greedy 策略选择动作 a。
                action, explored = select_action(Q[state], epsilon, generator)
                explore_count += int(explored)

                # ④ 环境执行动作；⑤ 返回 reward、next_state、done。
                result = env.step(action)

                # ⑥ 构造 TD 目标：终止状态之后没有未来价值。
                next_best = torch.tensor(0.0) if result.done.item() else Q[result.next_state].max()
                td_target = result.reward + gamma * next_best

                # ⑦ 计算 TD 误差；⑧ 原位更新当前状态—动作的 Q 值。
                td_error = td_target - Q[state, action]
                Q[state, action] += alpha * td_error

                # 把当前反馈计入本回合统计，并把 s' 作为下一步的 s。
                total_reward += result.reward
                td_error_sum += td_error.abs()
                state = result.next_state

                if result.done.item():
                    final_pos = env.state_to_pos[state.item()]
                    outcome = 1 if final_pos == env.goal else -1
                    break

            # 保存一个完整回合的训练指标。
            returns[episode-1] = total_reward
            lengths[episode-1] = step
            successes[episode-1] = outcome == 1
            epsilons[episode-1] = epsilon
            mean_abs_td[episode-1] = td_error_sum / step
            explore_ratios[episode-1] = explore_count / step
            outcomes[episode-1] = outcome

            # 探索率逐步下降，但最低保留 5% 探索。
            epsilon = max(epsilon_min, epsilon * epsilon_decay)

            # 保存关键时间点的 Q 表副本，用于展示策略如何形成。
            if episode in checkpoint_episodes:
                snapshots[episode] = Q.clone()

    history = {
        "returns": returns, "lengths": lengths, "successes": successes,
        "epsilons": epsilons, "mean_abs_td": mean_abs_td,
        "explore_ratios": explore_ratios, "outcomes": outcomes,
        "visits": visits, "snapshots": snapshots,
    }
    config = {
        "episodes": episodes, "alpha": alpha, "gamma": gamma,
        "epsilon_start": epsilon_start, "epsilon_min": epsilon_min,
        "epsilon_decay": epsilon_decay, "max_steps": max_steps, "seed": seed,
    }
    return Q, history, config


Q, history, train_config = train_q_learning(TorchGridWorld(DEVICE))

print(f"训练完成：Q.shape={tuple(Q.shape)}, dtype={Q.dtype}, device={Q.device}")
print(f"最后 100 回合平均奖励：{history['returns'][-100:].mean().item():.2f}")
print(f"最后 100 回合成功率：{history['successes'][-100:].float().mean().item():.1%}")
print(f"最后 100 回合平均步数：{history['lengths'][-100:].float().mean().item():.1f}")
print(f"最终探索率：{history['epsilons'][-1].item():.2f}")


## 9. 训练结果总览：是否真的学会了

训练曲线需要同时观察多个维度：

- 奖励上升、成功率上升：行为质量改善；
- 步数下降：路线更直接；
- ε 下降：从探索转向利用；
- TD 误差下降：Q 估计逐渐稳定；
- 探索动作占比下降：实际动作选择符合 ε 衰减预期。

单回合数据受探索影响会很抖，因此粗线使用 50 回合滑动平均。


In [ ]:
# ===== 9. 六指标训练仪表盘 =====
def rolling_mean(values, window=50):
    kernel = torch.ones(1, 1, window) / window
    smooth = F.conv1d(values.float().reshape(1, 1, -1), kernel).flatten()
    x = torch.arange(window, len(values)+1)
    return x, smooth

episodes = torch.arange(1, len(history["returns"])+1)
fig, axes = plt.subplots(2, 3, figsize=(15, 7.6), sharex=True)

specs = [
    ("returns", "回合奖励", "累计奖励：越高越好", C["blue"], 1.0),
    ("lengths", "步数", "回合长度：学会后缩短", C["orange"], 1.0),
    ("successes", "成功率 (%)", "成功率：到达终点比例", C["green"], 100.0),
    ("epsilons", "ε", "探索率：从探索到利用", C["coral"], 1.0),
    ("mean_abs_td", "|TD error|", "TD 误差：价值估计趋稳", C["teal"], 1.0),
    ("explore_ratios", "探索动作比例", "实际探索占比", C["gold"], 1.0),
]

for ax, (key, ylabel, title, color, scale) in zip(axes.flat, specs):
    raw = history[key].float() * scale
    ax.plot(episodes, raw, color=color, alpha=0.18, lw=0.7)
    x, smooth = rolling_mean(raw)
    ax.plot(x, smooth, color=color, lw=2.2)
    ax.set_title(title); ax.set_ylabel(ylabel); ax.set_xlabel("训练回合")
    ax.grid(alpha=0.15); ax.spines[["top", "right"]].set_visible(False)
    if key == "successes": ax.set_ylim(0, 103)

fig.suptitle("图 8｜训练全过程：行为改善、探索衰减、价值收敛", fontsize=15, weight="bold")
fig.tight_layout()
save_figure(fig, "08_training_dashboard")
plt.show()


In [ ]:
# ===== 10. 回合结局与探索/利用构成 =====
block = 50
n_blocks = len(history["outcomes"]) // block
outcome_blocks = history["outcomes"][:n_blocks*block].reshape(n_blocks, block)
goal_rate = (outcome_blocks == 1).float().mean(dim=1) * 100
trap_rate = (outcome_blocks == -1).float().mean(dim=1) * 100
timeout_rate = (outcome_blocks == 0).float().mean(dim=1) * 100
block_x = torch.arange(n_blocks) * block + block

fig, axes = plt.subplots(1, 2, figsize=(12.3, 4.5))

# 左：不同结局的堆叠比例
axes[0].bar(block_x, goal_rate, width=38, color=C["green"], label="到达终点")
axes[0].bar(block_x, trap_rate, width=38, bottom=goal_rate, color=C["trap"], label="进入陷阱")
axes[0].bar(block_x, timeout_rate, width=38, bottom=goal_rate+trap_rate, color=C["gray"], label="超时")
axes[0].set_ylim(0, 100); axes[0].set_xlabel("训练回合（每 50 回合一组）"); axes[0].set_ylabel("结局比例 (%)")
axes[0].set_title("回合结局如何随训练改变"); axes[0].legend(frameon=False, ncol=3, fontsize=8)

# 右：早中晚三个阶段的探索/利用比例
segments = {"早期\n1–100": slice(0,100), "中期\n351–450": slice(350,450), "后期\n701–800": slice(700,800)}
explore = torch.tensor([history["explore_ratios"][s].mean() for s in segments.values()]) * 100
exploit = 100 - explore
x = torch.arange(len(segments))
axes[1].bar(x, exploit, color=C["blue"], label="利用")
axes[1].bar(x, explore, bottom=exploit, color=C["gold"], label="探索")
axes[1].set_xticks(x, list(segments.keys())); axes[1].set_ylim(0,100); axes[1].set_ylabel("动作比例 (%)")
axes[1].set_title("实际动作选择：探索逐渐让位于利用"); axes[1].legend(frameon=False)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.14)
fig.suptitle("图 9｜智能体不仅成功更多，而且决策方式也发生变化", fontsize=14, weight="bold")
fig.tight_layout()
save_figure(fig, "09_outcomes_and_exploration")
plt.show()


## 10. Q 表如何逐渐长成策略

把每个状态的四个 Q 值取最大值，得到状态价值：

$$V(s)=\max_a Q(s,a)$$

箭头表示当前最大 Q 值对应的动作。终点奖励先影响附近状态，再通过 TD 目标逐步向更远状态传播。


In [ ]:
# ===== 11. 策略演化快照 =====
def values_to_grid(env, values, fill=torch.nan):
    grid = torch.full((env.rows, env.cols), fill, dtype=torch.float32)
    for state, pos in env.state_to_pos.items():
        grid[pos] = values[state].cpu()
    return grid

def add_policy_arrows(ax, env, q, min_value=1e-10):
    for state, pos in env.state_to_pos.items():
        if pos == env.goal or pos in env.traps:
            continue
        q_row = q[state]
        if q_row.abs().max().item() <= min_value:
            ax.text(pos[1], pos[0], "·", ha="center", va="center", color=C["gray"], fontsize=14)
        else:
            action = q_row.argmax().item()
            dr, dc = env.DELTAS[action]
            ax.arrow(pos[1], pos[0], dc*0.28, dr*0.28, head_width=0.13, head_length=0.11,
                     fc="white", ec="white", length_includes_head=True, linewidth=1.5, zorder=4)

snapshots = history["snapshots"]
checkpoint_eps = sorted(snapshots)
max_value = max(q.max().item() for q in snapshots.values())

fig, axes = plt.subplots(1, len(checkpoint_eps), figsize=(16.7, 3.8), sharex=True, sharey=True, layout="constrained")
for ax, episode in zip(axes, checkpoint_eps):
    q_snapshot = snapshots[episode]
    value_grid = values_to_grid(env, q_snapshot.max(dim=1).values)
    im = ax.imshow(value_grid, cmap="viridis", vmin=0, vmax=max_value)
    add_policy_arrows(ax, env, q_snapshot)
    for pos in env.walls:
        ax.add_patch(patches.Rectangle((pos[1]-0.5, pos[0]-0.5), 1, 1, facecolor=C["wall"], zorder=3))
    for pos in env.traps:
        ax.text(pos[1], pos[0], "陷阱", ha="center", va="center", color="white", weight="bold", fontsize=8, zorder=5)
    ax.text(env.goal[1], env.goal[0], "终点", ha="center", va="center", color="white", weight="bold", fontsize=8, zorder=5)
    ax.set_title(f"{episode} 回合"); ax.set_xticks([]); ax.set_yticks([])

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.72, pad=0.015, label="状态价值 max Q")
fig.suptitle("图 10｜价值从终点向外传播，策略箭头逐渐稳定", fontsize=14, weight="bold")
save_figure(fig, "10_policy_evolution")
plt.show()


## 11. 打开训练模型：完整 Q 表的四个动作切片

Q 表可以理解为四张叠在一起的地图：

- 第一张回答“在每个状态向上有多好”；
- 第二张回答“向右有多好”；
- 第三、第四张分别回答向下、向左。

在某个状态选择哪支箭头，就是比较该位置在四张图中的数值，取最大的动作。


In [ ]:
# ===== 12. 完整 Q 表热力图 =====
fig, axes = plt.subplots(2, 2, figsize=(10, 8.5), layout="constrained")
q_min, q_max = Q.min().item(), Q.max().item()

for action, ax in enumerate(axes.flat):
    action_grid = values_to_grid(env, Q[:, action])
    im = ax.imshow(action_grid, cmap="RdYlGn", vmin=q_min, vmax=q_max)
    for state, pos in env.state_to_pos.items():
        if pos not in env.traps and pos != env.goal:
            ax.text(pos[1], pos[0], f"{Q[state, action].item():.1f}", ha="center", va="center", fontsize=8)
    for pos in env.walls:
        ax.add_patch(patches.Rectangle((pos[1]-0.5, pos[0]-0.5), 1, 1, facecolor=C["wall"]))
        ax.text(pos[1], pos[0], "墙", ha="center", va="center", color="white")
    ax.set_title(f"动作 {action}：{env.ACTIONS[action]} {env.ARROWS[action]}")
    ax.set_xticks(range(env.cols)); ax.set_yticks(range(env.rows))

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.76, pad=0.02, label="Q(s,a)：越绿越值得选")
fig.suptitle("图 11｜训练后的 Q 表 = 24 个状态 × 4 个动作", fontsize=15, weight="bold")
save_figure(fig, "11_q_table_heatmaps")
plt.show()


In [ ]:
# ===== 13. 最终策略、状态价值与访问次数 =====
fig, axes = plt.subplots(1, 2, figsize=(12.2, 5.1))

# 左：最终状态价值 + 贪心动作
value_grid = values_to_grid(env, Q.max(dim=1).values)
im0 = axes[0].imshow(value_grid, cmap="viridis")
add_policy_arrows(axes[0], env, Q)
for pos in env.walls:
    axes[0].add_patch(patches.Rectangle((pos[1]-0.5, pos[0]-0.5), 1, 1, facecolor=C["wall"], zorder=3))
for pos in env.traps:
    axes[0].text(pos[1], pos[0], "陷阱", ha="center", va="center", color="white", weight="bold", zorder=5)
axes[0].text(env.goal[1], env.goal[0], "终点", ha="center", va="center", color="white", weight="bold", zorder=5)
format_grid_axes(axes[0], env, "最终价值 V(s) 与贪心策略")
fig.colorbar(im0, ax=axes[0], shrink=0.75, label="max Q")

# 右：访问频率不是价值，它只回答“训练时去过哪里”。
visit_grid = values_to_grid(env, history["visits"].float())
im1 = axes[1].imshow(visit_grid, cmap="Blues", norm=colors.LogNorm(vmin=1, vmax=max(2, history["visits"].max().item())))
for state, pos in env.state_to_pos.items():
    axes[1].text(pos[1], pos[0], str(history["visits"][state].item()), ha="center", va="center", fontsize=8, color=C["ink"])
for pos in env.walls:
    axes[1].add_patch(patches.Rectangle((pos[1]-0.5, pos[0]-0.5), 1, 1, facecolor=C["wall"]))
    axes[1].text(pos[1], pos[0], "墙", ha="center", va="center", color="white")
format_grid_axes(axes[1], env, "训练期间的状态访问次数（对数色阶）")
fig.colorbar(im1, ax=axes[1], shrink=0.75, label="访问次数")

fig.suptitle("图 12｜价值回答“哪里好”，访问次数回答“去过哪里”", fontsize=14, weight="bold")
fig.tight_layout()
save_figure(fig, "12_final_policy_and_visits")
plt.show()


## 12. 关闭探索：逐步验收训练结果

训练结束后令 $\varepsilon=0$，每一步都执行：

```python
action = Q[state].argmax()
```

这叫贪心评估。它只检验已学到的策略，不再故意尝试随机动作。


In [ ]:
# ===== 14. 贪心评估：保存每一步完整 transition =====
def greedy_rollout(env, Q, max_steps=50):
    state = env.reset()
    route = [env.state_to_pos[state.item()]]
    transitions = []

    with torch.no_grad():
        for t in range(max_steps):
            # 评估时关闭探索，直接选择最大 Q 值对应的动作。
            q_row = Q[state].clone()
            action = q_row.argmax()
            result = env.step(action)

            transitions.append({
                "t": t,
                "state": state.item(),
                "position": env.state_to_pos[state.item()],
                "q_row": q_row,
                "action": action.item(),
                "action_name": env.ACTIONS[action.item()],
                "reward": result.reward.item(),
                "next_state": result.next_state.item(),
                "next_position": env.state_to_pos[result.next_state.item()],
                "done": result.done.item(),
            })
            state = result.next_state
            route.append(env.state_to_pos[state.item()])
            if result.done.item():
                break
    return route, transitions


eval_env = TorchGridWorld(DEVICE)
route, transitions = greedy_rollout(eval_env, Q)
reached_goal = route[-1] == eval_env.goal
total_eval_reward = sum(step["reward"] for step in transitions)

print(f"到达终点：{reached_goal} | 步数：{len(transitions)} | 总奖励：{total_eval_reward:.1f}")


In [ ]:
# ===== 15. 路线地图 + 每步奖励时间线 =====
fig, axes = plt.subplots(1, 2, figsize=(12.3, 5.1), gridspec_kw={"width_ratios": [1.15, 1]})

# 左：空间轨迹
axes[0].imshow(base_grid(eval_env), cmap=GRID_CMAP, norm=GRID_NORM)
format_grid_axes(axes[0], eval_env, "贪心路线：数字表示时间顺序")
xs = torch.tensor([p[1] for p in route])
ys = torch.tensor([p[0] for p in route])
axes[0].plot(xs, ys, color=C["blue"], lw=4, alpha=0.75)
for i, (x, y) in enumerate(zip(xs, ys)):
    color = C["gold"] if i == len(xs)-1 else C["teal"]
    axes[0].scatter(x, y, s=360, color=color, edgecolor="white", lw=2, zorder=4)
    axes[0].text(x.item(), y.item(), str(i), ha="center", va="center",
                 color=C["ink"] if i == len(xs)-1 else "white", weight="bold", zorder=5)

# 右：每一步的奖励和累计奖励
step_rewards = torch.tensor([x["reward"] for x in transitions])
cumulative = step_rewards.cumsum(dim=0)
t = torch.arange(len(step_rewards))
bar_colors = [C["green"] if r > 0 else C["sky"] if r > -1 else C["trap"] for r in step_rewards]
axes[1].bar(t, step_rewards, color=bar_colors, label="即时奖励 r")
axes[1].plot(t, cumulative, color=C["ink"], marker="o", lw=2, label="累计奖励")
axes[1].axhline(0, color=C["gray"], lw=0.8)
axes[1].set_xticks(t, [f"t={i}" for i in t.tolist()])
axes[1].set_xlabel("决策步"); axes[1].set_ylabel("奖励")
axes[1].set_title("最后一步获得 +10，累计奖励跃升")
axes[1].legend(frameon=False); axes[1].grid(axis="y", alpha=0.15)
axes[1].spines[["top", "right"]].set_visible(False)

fig.suptitle("图 13｜训练结果验收：策略把状态—动作—奖励串成完整路线", fontsize=14, weight="bold")
fig.tight_layout()
save_figure(fig, "13_greedy_rollout")
plt.show()

# 逐步表格：讲解时可从上到下依次念出完整链条。
headers = ["t", "状态 s", "位置", "四个 Q 值", "动作 a", "奖励 r", "下一状态 s'", "下一位置", "done"]
rows = []
for item in transitions:
    rows.append([
        item["t"], item["state"], item["position"],
        "[" + ", ".join(f"{v:.1f}" for v in item["q_row"].tolist()) + "]",
        f"{item['action']}/{item['action_name']}", item["reward"], item["next_state"], item["next_position"], item["done"],
    ])
table = "<table style='border-collapse:collapse;font-size:13px'><tr>" + "".join(
    f"<th style='padding:6px 8px;border:1px solid #ddd;background:#264653;color:white'>{h}</th>" for h in headers
) + "</tr>" + "".join(
    "<tr>" + "".join(f"<td style='padding:6px 8px;border:1px solid #ddd;text-align:center'>{html.escape(str(v))}</td>" for v in row) + "</tr>"
    for row in rows
) + "</table>"
display(HTML(table))


## 13. 训练模型是什么：保存和加载 Q 表检查点

这个算法没有神经网络，因此“训练后的模型”不是一组网络权重，而是：

1. 学到的 `q_table`；
2. 状态与位置映射；
3. 动作含义；
4. 训练超参数和结果摘要；
5. PyTorch 版本与随机种子。

把这些内容保存为 `.pt` 检查点，之后不必重新训练即可加载并执行策略。


In [ ]:
# ===== 16. 保存、加载并验证训练模型 =====
MODEL_PATH = Path("torch_q_learning_checkpoint.pt")

checkpoint = {
    "q_table": Q.cpu(),
    "positions": env.positions,
    "actions": env.ACTIONS,
    "action_arrows": env.ARROWS,
    "train_config": train_config,
    "metrics": {
        "last100_mean_return": history["returns"][-100:].mean().item(),
        "last100_success_rate": history["successes"][-100:].float().mean().item(),
        "greedy_steps": len(transitions),
        "greedy_total_reward": total_eval_reward,
    },
    "seed": SEED,
    "torch_version": torch.__version__,
}

# torch.save 会把张量和配套 Python 元数据一起序列化。
torch.save(checkpoint, MODEL_PATH)

# 加载后验证 Q 表逐元素完全一致。
loaded = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
same_q = torch.equal(loaded["q_table"], Q.cpu())
file_kb = MODEL_PATH.stat().st_size / 1024

print(f"模型文件：{MODEL_PATH.resolve()}")
print(f"文件大小：{file_kb:.1f} KB | Q 表一致：{same_q}")
print("检查点字段：", list(loaded.keys()))
print("训练配置：", loaded["train_config"])
print("结果摘要：", loaded["metrics"])

# 检查点组成可视化
fig, ax = plt.subplots(figsize=(11.8, 4.8))
ax.set_xlim(0, 11.8); ax.set_ylim(0, 4.8); ax.axis("off")
ax.add_patch(patches.FancyBboxPatch((4.65, 1.62), 2.5, 1.55, boxstyle="round,pad=0.08",
             facecolor=C["teal"], edgecolor="white", lw=2))
ax.text(5.9, 2.62, ".pt 检查点", ha="center", weight="bold", color="white", fontsize=13)
ax.text(5.9, 2.10, f"{file_kb:.1f} KB\n可直接加载评估", ha="center", color="white")

items = [
    (0.3, 3.35, "Q 表", "24×4 float32\n训练得到的核心", C["blue"]),
    (0.3, 0.55, "状态/动作映射", "位置列表、动作名称\n解释张量索引", C["sky"]),
    (8.95, 3.35, "训练配置", "α、γ、ε、回合数\n保证可复现", C["gold"]),
    (8.95, 0.55, "指标与元数据", "成功率、步数、版本\n便于审计", C["coral"]),
]
for x0, y0, title, text_value, color in items:
    ax.add_patch(patches.FancyBboxPatch((x0, y0), 2.55, 1.05, boxstyle="round,pad=0.06",
                 facecolor=color, edgecolor="white"))
    ax.text(x0+1.275, y0+0.73, title, ha="center", weight="bold", color=C["ink"])
    ax.text(x0+1.275, y0+0.32, text_value, ha="center", va="center", fontsize=8.5, color=C["ink"])
    start = (x0+2.55, y0+0.52) if x0 < 5 else (x0, y0+0.52)
    end = (4.58, 2.4) if x0 < 5 else (7.22, 2.4)
    ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="->", lw=1.8, color=C["ink"]))

ax.set_title("图 14｜表格型 Q-learning 的训练模型 = Q 表 + 解释它所需的上下文", fontsize=14)
fig.tight_layout()
save_figure(fig, "14_checkpoint_anatomy")
plt.show()


## 14. 小实验：学习率如何影响训练

每次只改变 $\alpha$，保持其他设置和随机种子一致。比较最后 100 回合平均奖励、成功率和平均步数。

这也是一个练习模板：可以把变量换成 `gamma`、`epsilon_decay` 或奖励值。


In [ ]:
# ===== 17. 学习率对照实验 =====
alphas = [0.05, 0.15, 0.80]
alpha_results = []
for alpha_value in alphas:
    q_a, h_a, _ = train_q_learning(TorchGridWorld(DEVICE), alpha=alpha_value, seed=SEED)
    route_a, steps_a = greedy_rollout(TorchGridWorld(DEVICE), q_a)
    alpha_results.append({
        "alpha": alpha_value,
        "return": h_a["returns"][-100:].mean().item(),
        "success": h_a["successes"][-100:].float().mean().item() * 100,
        "length": h_a["lengths"][-100:].float().mean().item(),
        "greedy_steps": len(steps_a),
    })

fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.9))
x = torch.arange(len(alphas))
metrics = [
    ("return", "最后100回合平均奖励", C["blue"]),
    ("success", "最后100回合成功率 (%)", C["green"]),
    ("length", "最后100回合平均步数", C["orange"]),
]
for ax, (key, title, color) in zip(axes, metrics):
    values = torch.tensor([r[key] for r in alpha_results])
    bars = ax.bar(x, values, color=color, width=0.58)
    ax.set_xticks(x, [str(a) for a in alphas]); ax.set_xlabel("学习率 α"); ax.set_title(title)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x()+bar.get_width()/2, value.item()+0.02*max(1, values.max().item()),
                f"{value.item():.1f}", ha="center", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.14)
fig.suptitle("图 15｜一次只改一个变量：学习率对结果的影响", fontsize=14, weight="bold")
fig.tight_layout()
save_figure(fig, "15_alpha_comparison")
plt.show()

for row in alpha_results:
    print(row)


## 15. 练习、常见误区与扩展

### 练习

1. 把普通移动奖励从 `-0.1` 改成 `0`，观察路线长度是否仍被强烈约束。
2. 把 `gamma` 改成 `0.5`，比较离终点较远状态的 Q 值。
3. 把 `epsilon_min` 改成 `0`，比较训练末期曲线是否更平滑。
4. 给动作加入 10% 打滑概率，观察确定性策略如何变化。

答案脚手架：复制 `TorchGridWorld`，把四种奖励改为构造参数；然后复用 `train_q_learning()` 和现有绘图函数。

### 常见误区

- **把奖励当成状态标签**：奖励评价刚发生的转移，不是简单给格子打分。
- **终止后继续 bootstrap**：`done=True` 时未来价值必须置零。
- **让 Q 表参与自动微分**：表格型 Q-learning 直接赋值，不需要 `requires_grad`、loss、`backward()` 或 optimizer。
- **小 Q 表强行放 GPU**：频繁 `.item()` 会同步设备；本例在 CPU 更清晰也更快。
- **访问频率等同于价值**：访问次数只代表去过多少次，价值由奖励和未来回报决定。
- **只看最后一个回合**：探索会造成偶然失败，应看滑动平均和关闭探索后的独立评估。

### 从这里到 DQN

当前模型是完整 Q 表：每个状态—动作组合都有一个明确单元格。当状态空间大到无法枚举时，用神经网络输入状态、输出四个动作的 Q 值，这才进入 DQN。


## 16. 一页讲解总结

```text
环境 reset() 给出状态 s
          ↓
查看 Q[s]，用 ε-greedy 选动作 a
          ↓
环境 step(a) 返回 reward、next_state、done
          ↓
TD target = reward + gamma × max(Q[next_state])
          ↓
Q[s,a] 向 TD target 移动 alpha 的比例
          ↓
next_state 成为新的 state；若 done=False 则继续
          ↓
很多回合后，argmax Q[s] 形成最终策略
          ↓
保存 Q 表和配套映射为 torch_q_learning_checkpoint.pt
```

最值得记住的四句话：

1. **环境定义世界如何响应。**
2. **状态和动作定义智能体看到什么、能做什么。**
3. **奖励定义什么行为值得学习。**
4. **Q-learning 用“眼前奖励 + 最好的未来”更新长期价值。**

配套文件：

- `minimal_q_learning_pytorch.py`：带关键注释的纯 PyTorch 单文件版；
- `torch_q_learning_checkpoint.pt`：训练后的 Q 表检查点；
- `figures_pytorch_full/`：本 notebook 导出的全部 PNG/PDF 图表。
